# 🔁 Day 5C — RAG Query Pipeline: Retrieve, Augment, Generate
**Agentic AI for Tax Technologists · 12-Day Program**

---

## 📖 Where We Are

| Notebook | What we built |
|---|---|
| 5A | Understood embeddings + similarity — the maths behind RAG |
| 5B | Built the ChromaDB vector store — 20 tax chunks indexed and queryable |
| **5C** | **Wire retrieval into generation — the complete RAG pipeline** |

**This notebook completes the Day 5 arc.** By the end:
- You have a working `ask_with_rag()` function
- Every answer cites its source section
- You can see side-by-side: the same question without RAG vs with RAG
- Results exported for Day 6 reference

---
## ⏱️ Time: 30 minutes

---
## ⚙️ Step 1: Install & Import

In [1]:
%pip install openai chromadb numpy --quiet


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import re
import numpy as np
import chromadb
import datetime
from getpass import getpass
from openai import AzureOpenAI

print("✅ Imports OK")

✅ Imports OK


---
## 🔑 Step 2: Configure Azure OpenAI

In [4]:
AZURE_OPENAI_ENDPOINT       = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY        = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME       = input("Chat deployment name (e.g. gpt-4o): ").strip()
AZURE_EMBEDDING_DEPLOYMENT  = input("Embeddings deployment name (e.g. text-embedding-ada-002): ").strip()
AZURE_API_VERSION           = "2024-08-01-preview"

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)
print("✅ Azure OpenAI client initialised successfully!")

Endpoint (e.g. https://xxx.openai.azure.com/):  https://agentic-ai-tt.openai.azure.com/
API Key:  ········
Chat deployment name (e.g. gpt-4o):  gpt-4o
Embeddings deployment name (e.g. text-embedding-ada-002):  text-embedding-ada-002


✅ Azure OpenAI client initialised successfully!


---
## 📦 Step 3: Rebuild the Vector Store (self-contained)

This notebook is self-contained — it rebuilds the knowledge base so you don't need 5B open in parallel.

In [ ]:
# ── Tax knowledge chunks (same 20 as Notebook 5B) ────────────────────────────
TAX_KNOWLEDGE_CHUNKS = [
    {"id":"gst_export_001","text":"Export of services is treated as zero-rated supply under Section 16(1) of the IGST Act 2017. A registered supplier may export without payment of IGST if a Letter of Undertaking (LUT) has been filed under Rule 96A. Alternatively, the supplier may export on payment of IGST and claim a refund under Section 54 of the CGST Act. LUT is valid for the entire financial year from the date of filing.","metadata":{"source":"IGST Act 2017","section":"16(1)","type":"legislation","topic":"exports","year":"2017"}},
    {"id":"gst_export_002","text":"Place of supply for export of services is determined under Section 13 of the IGST Act. When the recipient is located outside India, the place of supply is the location of the recipient. IT/ITES services provided to overseas clients qualify as export of services, subject to: (a) supplier is in India, (b) recipient is outside India, (c) payment received in convertible foreign exchange, (d) supplier and recipient are not merely establishments of the same person.","metadata":{"source":"IGST Act 2017","section":"13","type":"legislation","topic":"exports","year":"2017"}},
    {"id":"gst_notification_01_2023","text":"Notification No. 01/2023-Integrated Tax (Rate) clarifies the GST rate schedule for IT and ITES services. Cloud computing services, SaaS subscriptions, and data processing services supplied by Indian entities to overseas recipients qualify as export of services and are zero-rated if LUT is filed. If LUT is not filed, IGST at 18% applies and refund may be claimed. Domestic B2B supply of these services attracts 18% GST.","metadata":{"source":"Notification 01/2023-IT(Rate)","section":"4","type":"notification","topic":"exports","year":"2023"}},
    {"id":"gst_composition_001","text":"Section 10 of the CGST Act 2017 provides for the composition scheme. Registered suppliers with aggregate turnover not exceeding Rs 1.5 crore may opt for composition levy. Composition dealers pay tax at 1% for traders, 2% for manufacturers, 5% for restaurants. Cannot collect tax from recipients, cannot issue tax invoices, and cannot avail input tax credit. Must issue a Bill of Supply instead.","metadata":{"source":"CGST Act 2017","section":"10","type":"legislation","topic":"composition_scheme","year":"2017"}},
    {"id":"gst_itc_001","text":"Input Tax Credit (ITC) under Section 16 of the CGST Act can be claimed if: (a) the supplier has filed returns and paid the tax, (b) the recipient holds a valid tax invoice, (c) the goods or services have been received, and (d) the tax has been paid to the government. Section 17(5) lists blocked credits including motor vehicles for personal use, food and beverages, beauty treatment, health services, and club memberships.","metadata":{"source":"CGST Act 2017","section":"16","type":"legislation","topic":"input_tax_credit","year":"2017"}},
    {"id":"gst_rcm_001","text":"Reverse Charge Mechanism (RCM) under Section 9(3) of the CGST Act requires the recipient to pay GST instead of the supplier in notified categories. Key RCM notifications: legal services by an advocate, services by a director to a company, goods transport agency (GTA) services, import of services from overseas suppliers, and security services supplied by an individual to a body corporate.","metadata":{"source":"CGST Act 2017","section":"9(3)","type":"legislation","topic":"reverse_charge","year":"2017"}},
    {"id":"gst_registration_001","text":"GST registration is mandatory when aggregate turnover exceeds Rs 20 lakh for service providers (Rs 10 lakh in special category states) and Rs 40 lakh for goods suppliers. Inter-state suppliers must register regardless of turnover. E-commerce operators must register regardless of turnover. Voluntary registration is permitted below the threshold.","metadata":{"source":"CGST Act 2017","section":"22","type":"legislation","topic":"registration","year":"2017"}},
    {"id":"gst_gstr1_001","text":"GSTR-1 is the return for outward supplies. Monthly filers with turnover above Rs 5 crore must file by the 11th of the following month. Quarterly filers under QRMP scheme file by the 13th of the month following each quarter. Late filing attracts Rs 50/day (Rs 20/day for nil returns).","metadata":{"source":"CGST Rules 2017","section":"Rule 59","type":"rule","topic":"filing","year":"2017"}},
    {"id":"gst_gstr3b_001","text":"GSTR-3B is the monthly summary return for payment of GST liability. Monthly taxpayers file by the 20th of the following month. Under QRMP scheme, GSTR-3B is filed quarterly by the 22nd for Category-I states and 24th for Category-II states. Payment must accompany filing.","metadata":{"source":"CGST Rules 2017","section":"Rule 61","type":"rule","topic":"filing","year":"2017"}},
    {"id":"tds_194c_001","text":"Section 194C requires TDS on payments to contractors and sub-contractors. TDS rate is 1% for individuals and HUFs, and 2% for companies and firms. TDS is deducted when a single payment exceeds Rs 30,000 or aggregate payments in a financial year exceed Rs 1,00,000. Section 194C covers advertising contracts, catering, broadcasting, and carriage of goods.","metadata":{"source":"Income Tax Act 1961","section":"194C","type":"legislation","topic":"tds","year":"1961"}},
    {"id":"tds_194j_001","text":"Section 194J covers TDS on fees for professional or technical services. TDS rate is 10% for professional services (lawyers, doctors, engineers, CAs) and 2% for technical services. The threshold is Rs 30,000 per annum per payee. Professional services include medical, legal, engineering, and management consultancy.","metadata":{"source":"Income Tax Act 1961","section":"194J","type":"legislation","topic":"tds","year":"1961"}},
    {"id":"tds_return_001","text":"TDS returns must be filed quarterly. Form 24Q covers TDS on salaries, Form 26Q covers TDS on non-salary payments to residents, Form 27Q covers TDS on payments to non-residents. Due dates: Q1 July 31; Q2 October 31; Q3 January 31; Q4 May 31. TDS certificate Form 16 (salary) and Form 16A (non-salary) must be issued within 15 days of the due date of filing.","metadata":{"source":"Income Tax Rules 1962","section":"Rule 31A","type":"rule","topic":"tds","year":"1962"}},
    {"id":"corp_tax_rates_001","text":"Corporate income tax rates for Indian domestic companies: 25% for companies with turnover not exceeding Rs 400 crore; 30% for others. Section 115BAB provides 15% for new domestic manufacturing companies incorporated after October 1, 2019. Effective rate including surcharge and cess is approximately 25.17% for companies under the 25% slab.","metadata":{"source":"Income Tax Act 1961","section":"115BAB","type":"legislation","topic":"corporate_tax","year":"2019"}},
    {"id":"transfer_pricing_001","text":"Transfer pricing under Sections 92 to 92F requires international transactions between associated enterprises to be at arm's length price. Methods: CUP, resale price, cost plus, TNMM, and profit split. Taxpayers with international transactions exceeding Rs 1 crore must obtain a Transfer Pricing Report in Form 3CEB from a CA. Due date: October 31.","metadata":{"source":"Income Tax Act 1961","section":"92-92F","type":"legislation","topic":"transfer_pricing","year":"1961"}},
    {"id":"advance_tax_001","text":"Advance tax is payable in four instalments when estimated tax liability exceeds Rs 10,000. Instalments: 15% by June 15, 45% cumulative by September 15, 75% cumulative by December 15, 100% by March 15. Interest under Section 234B applies at 1% per month for default. Senior citizens with no business income are exempt from advance tax.","metadata":{"source":"Income Tax Act 1961","section":"208-219","type":"legislation","topic":"advance_tax","year":"1961"}},
    {"id":"section_80c_001","text":"Section 80C allows deductions up to Rs 1,50,000 per year. Eligible: LIC premiums, PPF, ELSS, NSC, 5-year FDs, tuition fees for up to two children, EPF, home loan principal repayment. Available only under the old tax regime. Under the new regime (Section 115BAC), Section 80C deductions are not available.","metadata":{"source":"Income Tax Act 1961","section":"80C","type":"legislation","topic":"deductions","year":"1961"}},
    {"id":"new_tax_regime_001","text":"New tax regime under Section 115BAC (FY 2025-26): Nil up to Rs 3 lakh, 5% for Rs 3-7 lakh, 10% for Rs 7-10 lakh, 15% for Rs 10-12 lakh, 20% for Rs 12-15 lakh, 30% above Rs 15 lakh. Standard deduction Rs 75,000. Rebate under Section 87A: zero tax for income up to Rs 7 lakh. No 80C or HRA deductions. New regime is the default from AY 2024-25.","metadata":{"source":"Income Tax Act 1961","section":"115BAC","type":"legislation","topic":"income_tax_slabs","year":"2023"}},
    {"id":"ltcg_001","text":"LTCG on listed equity and equity-oriented mutual funds exceeding Rs 1 lakh is taxed at 10% without indexation under Section 112A. Holding period for equity: more than 12 months. For unlisted shares and immovable property: more than 24 months. Short-term capital gains on listed equity are taxed at 15% under Section 111A.","metadata":{"source":"Income Tax Act 1961","section":"112A","type":"legislation","topic":"capital_gains","year":"1961"}},
    {"id":"itr_filing_001","text":"Income tax return filing due dates: July 31 for individuals and HUFs; October 31 for tax audit cases and companies; November 30 for transfer pricing cases. Belated return by December 31 with late fee under Section 234F (Rs 5,000 for income above Rs 5 lakh). Revised return also by December 31.","metadata":{"source":"Income Tax Act 1961","section":"139","type":"legislation","topic":"filing","year":"1961"}},
    {"id":"gst_works_contract_001","text":"Works contract under GST is treated as a supply of service. GST rates: 18% general; 12% for construction of affordable housing, government civil structures, and roads. ITC on works contracts is allowed to the contractor against output liability, subject to Section 17(5)(c) restrictions on immovable property construction.","metadata":{"source":"Notification 11/2017-CT(Rate)","section":"Entry 3","type":"notification","topic":"works_contract","year":"2017"}},
]

print(f"✅ {len(TAX_KNOWLEDGE_CHUNKS)} knowledge chunks loaded")

In [ ]:
def get_embedding(text):
    """Get embedding vector from Azure OpenAI."""
    response = client.embeddings.create(
        model = AZURE_EMBEDDING_DEPLOYMENT,
        input = text
    )
    return response.data[0].embedding


print("Building vector store...")
chroma_client  = chromadb.Client()
tax_collection = chroma_client.create_collection(
    name="tax_knowledge", metadata={"hnsw:space": "cosine"}
)

for i, chunk in enumerate(TAX_KNOWLEDGE_CHUNKS, 1):
    vec = get_embedding(chunk["text"])
    tax_collection.add(
        ids        = [chunk["id"]],
        documents  = [chunk["text"]],
        embeddings = [vec],
        metadatas  = [{k: str(v) for k, v in chunk["metadata"].items()}],
    )
    print(f"  [{i:02d}/{len(TAX_KNOWLEDGE_CHUNKS)}]", end="\r")

print(f"\n✅ Vector store ready — {tax_collection.count()} chunks indexed")

---
## 🔧 Step 4: Build the Core RAG Function

In [ ]:
RAG_SYSTEM_PROMPT = (
    "You are a senior tax consultant at a Big4 firm.\n"
    "Answer questions using ONLY the context provided below.\n"
    "CRITICAL RULES:\n"
    "1. Every factual statement must cite its source section in square brackets, e.g. [IGST Act S.16(1)] or [Notification 01/2023-IT(Rate) S.4].\n"
    "2. If the answer is not in the context, say 'This specific question is not covered in the retrieved sections. Please consult the full legislation.'\n"
    "3. Never use information from your training data — only the context below.\n"
    "4. Keep answers concise: 3–6 sentences maximum."
)

BASELINE_SYSTEM_PROMPT = (
    "You are a senior tax consultant at a Big4 firm.\n"
    "Answer Indian tax questions accurately and concisely in 3–6 sentences."
)


def retrieve(query, n_results=3):
    """Retrieve top-k chunks from ChromaDB by semantic similarity."""
    q_vec = get_embedding(query)
    results = tax_collection.query(query_embeddings=[q_vec], n_results=n_results)
    chunks = []
    for i in range(len(results["ids"][0])):
        chunks.append({
            "id":       results["ids"][0][i],
            "text":     results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "similarity": round(1 - results["distances"][0][i], 3),
        })
    return chunks


def format_context(chunks):
    """Format retrieved chunks into a context block for the prompt."""
    lines = ["RETRIEVED CONTEXT:\n"]
    for i, chunk in enumerate(chunks, 1):
        m = chunk["metadata"]
        lines.append(f"[Source {i}: {m.get('source','?')} §{m.get('section','?')}]")
        lines.append(chunk["text"])
        lines.append("")
    return "\n".join(lines)


def ask_without_rag(question):
    """Answer using only model training knowledge — no retrieval."""
    response = client.chat.completions.create(
        model       = AZURE_DEPLOYMENT_NAME,
        messages    = [
            {"role": "system", "content": BASELINE_SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature = 0.0,
        max_tokens  = 400,
    )
    return response.choices[0].message.content


def ask_with_rag(question, n_chunks=3, verbose=False):
    """
    Full RAG pipeline:
    1. Embed the question
    2. Retrieve top-k chunks from ChromaDB
    3. Inject chunks as context into the system prompt
    4. Generate grounded, cited answer via Azure OpenAI
    """
    # Step 1 & 2: Retrieve
    chunks = retrieve(question, n_results=n_chunks)

    if verbose:
        print("  Retrieved chunks:")
        for c in chunks:
            print(f"    [{c['similarity']:.3f}] {c['id']}")

    # Step 3: Format context
    context = format_context(chunks)

    # Step 4: Generate
    augmented_prompt = RAG_SYSTEM_PROMPT + "\n\n" + context
    response = client.chat.completions.create(
        model       = AZURE_DEPLOYMENT_NAME,
        messages    = [
            {"role": "system", "content": augmented_prompt},
            {"role": "user",   "content": question},
        ],
        temperature = 0.0,
        max_tokens  = 500,
    )
    return response.choices[0].message.content, chunks


print("✅ RAG pipeline functions ready")

---
## 🧪 Step 5: Test on 5 Tax Questions

In [ ]:
TEST_QUESTIONS = [
    "What GST rate applies to cloud software exported to a US company with LUT filed?",
    "A company avails legal services from a senior advocate. Does GST apply and who pays?",
    "What are the income tax slabs under the new tax regime for FY 2025-26?",
    "When is GSTR-3B due for a monthly taxpayer? What does it cover?",
    "Can I claim ITC on food expenses incurred for office employees under GST?",
]

print("Running RAG pipeline on 5 questions...\n")

rag_results = []
for q in TEST_QUESTIONS:
    answer, chunks = ask_with_rag(q, n_chunks=3, verbose=True)
    rag_results.append({"question": q, "answer": answer, "chunks_used": [c["id"] for c in chunks]})
    print(f"  Q: {q}")
    print(f"  A: {answer}")
    print()

---
## 🆚 Step 6: Side-by-Side — Without RAG vs With RAG

In [ ]:
# Choose the most interesting question for comparison
COMPARISON_QUESTION = "What GST rate applies to cloud software exported to a US company with LUT filed?"

print("\n" + "="*70)
print(f"QUESTION: {COMPARISON_QUESTION}")
print("="*70)

print("\n--- WITHOUT RAG (model training knowledge only) ---")
ans_baseline = ask_without_rag(COMPARISON_QUESTION)
print(ans_baseline)

print("\n--- WITH RAG (retrieved from vector store) ---")
ans_rag, chunks_used = ask_with_rag(COMPARISON_QUESTION, verbose=True)
print(ans_rag)

print("\n" + "="*70)
print("ANALYSIS")
print("="*70)
print()
has_citation = "[" in ans_rag and "]" in ans_rag
cites_section = any(s in ans_rag for s in ["S.16", "Section 16", "Notification", "IGST"])
baseline_has_citation = "[" in ans_baseline

print(f"WITHOUT RAG — contains citation: {baseline_has_citation}")
print(f"WITH RAG    — contains citation: {has_citation}")
print(f"WITH RAG    — cites section/source: {cites_section}")
print()
print("Chunks used:")
for c in chunks_used:
    print(f"  [{c['similarity']:.3f}] {c['id']} — {c['metadata'].get('source','?')} §{c['metadata'].get('section','?')}")

---
## 📊 Step 7: Citation Quality Audit — All 5 Questions

In [ ]:
print("CITATION QUALITY AUDIT")
print("-" * 60)
print(f"{'Q#':<4} {'Has citation':>14} {'Cites section':>15} {'Chunks used':>14}")
print("-" * 60)

citation_patterns = [r"\[.*?\]", r"Section \d+", r"Notification", r"Rule \d+"]

for i, r in enumerate(rag_results, 1):
    has_cit  = any(re.search(p, r["answer"]) for p in citation_patterns)
    n_chunks = len(r["chunks_used"])
    print(f"  Q{i:<3} {'YES' if has_cit else 'NO':>14} {'N/A':>15} {n_chunks:>14}")

print()
print("Note: Answers that cite retrieved sections are grounded.")
print("Any answer that cannot cite a section should trigger human review.")

---
## 💾 Step 8: Export Results

In [ ]:
output = {
    "notebook":    "Day5C_RAG_Pipeline",
    "exported_at": datetime.datetime.now().isoformat(),
    "model":       AZURE_DEPLOYMENT_NAME,
    "embedding_model": AZURE_EMBEDDING_DEPLOYMENT,
    "knowledge_base_size": len(TAX_KNOWLEDGE_CHUNKS),
    "results": rag_results,
    "comparison": {
        "question":     COMPARISON_QUESTION,
        "without_rag":  ans_baseline,
        "with_rag":     ans_rag,
        "chunks_used": [c["id"] for c in chunks_used],
    }
}

with open("day5_rag_results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("✅ Results saved to day5_rag_results.json")

---
## 🎓 The Full Day 5 Arc

```
Day 2:  ReAct agent  →  tool calls + short-term memory
Day 3:  CoT + few-shot  →  structured outputs from clean questions
Day 4:  Function calling  →  extraction from unstructured documents
Day 5:  RAG  →  agent finds the right document before answering

Day 5A: embeddings — meaning encoded as numbers
Day 5B: ChromaDB — 20 tax chunks indexed and queryable
Day 5C: RAG pipeline — retrieve → augment → generate with citations
```

**What you have now:** a semantic search engine over a tax knowledge base that generates grounded, cited answers.

---

## ➡️ Day 6: Advanced RAG & Source Citation

Day 5 retrieves. Day 6 makes retrieval production-grade:
- Extract page numbers and section numbers from document metadata
- Format citations properly (not just in brackets — in a reference list)
- Multi-document retrieval with deduplication
- RAG evaluation: relevance score, faithfulness score, answer correctness
- Test against a golden Q&A dataset to measure retrieval quality